In [1]:
import csv
import re
from collections import Counter, defaultdict
from pathlib import Path
from pprint import pprint

from IPython.display import display, Markdown


def resolve_project_root() -> Path:
    candidate = Path.cwd()
    expected = candidate / 'Datos' / 'csv' / 'establecimientos_diversificado_consolidado.csv'
    if expected.exists():
        return candidate
    return Path(r'c:/Users/aeeh2/Documents/Universidad/Semestre 8/Data Science/PRY1-DS')


base_dir = resolve_project_root()
csv_path = base_dir / 'Datos' / 'csv' / 'establecimientos_diversificado_consolidado.csv'
raw_dir = base_dir / 'Datos' / 'raw'
diagnostics_dir = base_dir / 'Datos' / 'diagnosticos'

with csv_path.open(newline='', encoding='utf-8', errors='replace') as handle:
    reader = csv.DictReader(handle)
    rows = [{key: (value or '').strip() for key, value in row.items()} for row in reader]

headers = reader.fieldnames or []
row_count = len(rows)
column_count = len(headers)
raw_department_catalog = {path.stem.strip().upper() for path in raw_dir.glob('*.xls')}


EXPECTED_CATEGORICAL_DOMAINS = {
    'NIVEL': {'DIVERSIFICADO'},
    'AREA': {'URBANA', 'RURAL'},
}


PLACEHOLDERS = {'', '-', '--', '---', '----', 'SIN DATO', 'N/A', 'NA'}
CODE_PATTERN = re.compile(r'^\d{2}-\d{2}-\d{4}-\d{2}$')
PHONE_PATTERN = re.compile(r'^[0-9]+$')


def normalize_text(value: str) -> str:
    return re.sub(r'\s+', ' ', (value or '').strip())


def is_blank(value: str) -> bool:
    return normalize_text(value) == ''


def normalize_upper(value: str) -> str:
    return normalize_text(value).upper()


def value_counter(column_name: str) -> Counter:
    return Counter(normalize_text(row.get(column_name, '')) for row in rows)


def print_table(title: str, records: list[dict], limit: int = 10) -> None:
    display(Markdown(f'### {title}'))
    if not records:
        print('Sin resultados')
        return
    preview = records[:limit]
    columns = list(preview[0].keys())
    widths = {column: len(column) for column in columns}
    for record in preview:
        for column in columns:
            widths[column] = max(widths[column], len(str(record.get(column, ''))))
    print(' | '.join(column.ljust(widths[column]) for column in columns))
    print(' | '.join('-' * widths[column] for column in columns))
    for record in preview:
        print(' | '.join(str(record.get(column, '')).ljust(widths[column]) for column in columns))
    if len(records) > limit:
        print(f'... {len(records) - limit} filas adicionales')


def export_table_csv(records: list[dict], filename: str) -> Path:
    diagnostics_dir.mkdir(parents=True, exist_ok=True)
    path = diagnostics_dir / filename
    if not records:
        with path.open('w', newline='', encoding='utf-8') as handle:
            handle.write('')
        return path
    with path.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(records[0].keys()))
        writer.writeheader()
        writer.writerows(records)
    return path


print(f'Ruta: {csv_path}')
print(f'Forma: {row_count:,} filas x {column_count} columnas')
print('Columnas:')
for header in headers:
    print(f'- {header}')
print('\nPrimeras 5 filas:')
pprint(rows[:5], width=140)
print(f'\nCatalogo local de departamentos detectado: {len(raw_department_catalog)} valores')

Ruta: C:\Users\aeeh2\Documents\Universidad\Semestre 8\Data Science\PRY1-DS\Datos\csv\establecimientos_diversificado_consolidado.csv
Forma: 11,867 filas x 17 columnas
Columnas:
- CODIGO
- DISTRITO
- DEPARTAMENTO
- MUNICIPIO
- ESTABLECIMIENTO
- DIRECCION
- TELEFONO
- SUPERVISOR
- DIRECTOR
- NIVEL
- SECTOR
- AREA
- STATUS
- MODALIDAD
- JORNADA
- PLAN
- DEPARTAMENTAL

Primeras 5 filas:
[{'AREA': 'URBANA',
  'CODIGO': '16-01-0137-46',
  'DEPARTAMENTAL': 'ALTA VERAPAZ',
  'DEPARTAMENTO': 'ALTA VERAPAZ',
  'DIRECCION': '6A. AVENIDA 1-15 ZONA 4',
  'DIRECTOR': '--',
  'DISTRITO': '16-006',
  'ESTABLECIMIENTO': 'INSTITUTO MIXTO NOCTURNO FRANCISCO MARROQUIN',
  'JORNADA': 'NOCTURNA',
  'MODALIDAD': 'MONOLINGUE',
  'MUNICIPIO': 'COBAN',
  'NIVEL': 'DIVERSIFICADO',
  'PLAN': 'DIARIO(REGULAR)',
  'SECTOR': 'PRIVADO',
  'STATUS': 'CERRADA DEFINITIVAMENTE',
  'SUPERVISOR': 'JORGE EDUARDO PAQUE LAZARO',
  'TELEFONO': ''},
 {'AREA': 'URBANA',
  'CODIGO': '16-01-0138-46',
  'DEPARTAMENTAL': 'ALTA VERAPA

## 1. Load and Inspect the Dataset

Esta sección carga el consolidado de establecimientos educativos y deja visibles las primeras señales de calidad: tamaño, estructura, valores nulos y duplicados.

In [2]:
missing_report = []
for column in headers:
    missing_count = sum(1 for row in rows if is_blank(row.get(column, '')))
    missing_report.append({
        'column': column,
        'missing_count': missing_count,
        'missing_pct': round((missing_count / row_count) * 100, 2),
        'non_missing_count': row_count - missing_count,
    })

missing_report.sort(key=lambda record: (-record['missing_count'], record['column']))
print_table('Reporte de valores faltantes', missing_report, limit=17)
missing_export = export_table_csv(missing_report, 'reporte_faltantes.csv')
print(f'Archivo exportado: {missing_export}')

### Reporte de valores faltantes


column          | missing_count | missing_pct | non_missing_count
--------------- | ------------- | ----------- | -----------------
DIRECTOR        | 1733          | 14.6        | 10134            
TELEFONO        | 946           | 7.97        | 10921            
SUPERVISOR      | 535           | 4.51        | 11332            
DISTRITO        | 532           | 4.48        | 11335            
DIRECCION       | 76            | 0.64        | 11791            
ESTABLECIMIENTO | 5             | 0.04        | 11862            
AREA            | 0             | 0.0         | 11867            
CODIGO          | 0             | 0.0         | 11867            
DEPARTAMENTAL   | 0             | 0.0         | 11867            
DEPARTAMENTO    | 0             | 0.0         | 11867            
JORNADA         | 0             | 0.0         | 11867            
MODALIDAD       | 0             | 0.0         | 11867            
MUNICIPIO       | 0             | 0.0         | 11867            
NIVEL     

In [3]:
full_duplicates = 0
seen_rows = set()
for row in rows:
    signature = tuple(row.get(column, '') for column in headers)
    if signature in seen_rows:
        full_duplicates += 1
    else:
        seen_rows.add(signature)

codigo_counter = Counter(row.get('CODIGO', '') for row in rows if not is_blank(row.get('CODIGO', '')))
codigo_duplicates = [
    {'CODIGO': codigo, 'repeticiones': count}
    for codigo, count in sorted(codigo_counter.items())
    if count > 1
]

invalid_codigo_rows = [
    {
        'row_number': index,
        'CODIGO': row.get('CODIGO', ''),
        'DISTRITO': row.get('DISTRITO', ''),
        'DEPARTAMENTO': row.get('DEPARTAMENTO', ''),
        'MUNICIPIO': row.get('MUNICIPIO', ''),
    }
    for index, row in enumerate(rows, start=2)
    if not CODE_PATTERN.match(row.get('CODIGO', ''))
]

duplicate_summary = [
    {'tipo': 'duplicados exactos', 'cantidad': full_duplicates},
    {'tipo': 'CODIGO repetido', 'cantidad': sum(item['repeticiones'] - 1 for item in codigo_duplicates)},
    {'tipo': 'CODIGO invalido', 'cantidad': len(invalid_codigo_rows)},
]

print_table('Resumen de duplicados e identificadores invalidados', duplicate_summary, limit=10)
print_table('CODIGO repetidos', codigo_duplicates, limit=20)
print_table('Registros con CODIGO invalido', invalid_codigo_rows, limit=20)

duplicates_export = export_table_csv(codigo_duplicates, 'reporte_codigos_repetidos.csv')
invalid_codigo_export = export_table_csv(invalid_codigo_rows, 'reporte_codigos_invalidos.csv')
print(f'Archivo exportado: {duplicates_export}')
print(f'Archivo exportado: {invalid_codigo_export}')

### Resumen de duplicados e identificadores invalidados


### CODIGO repetidos


### Registros con CODIGO invalido


tipo               | cantidad
------------------ | --------
duplicados exactos | 0       
CODIGO repetido    | 0       
CODIGO invalido    | 0       
Sin resultados
Sin resultados
Archivo exportado: C:\Users\aeeh2\Documents\Universidad\Semestre 8\Data Science\PRY1-DS\Datos\diagnosticos\reporte_codigos_repetidos.csv
Archivo exportado: C:\Users\aeeh2\Documents\Universidad\Semestre 8\Data Science\PRY1-DS\Datos\diagnosticos\reporte_codigos_invalidos.csv


In [4]:
placeholder_rows = []
phone_rows = []
text_columns = ['SUPERVISOR', 'DIRECTOR', 'ESTABLECIMIENTO', 'DIRECCION']

for index, row in enumerate(rows, start=2):
    director = normalize_upper(row.get('DIRECTOR', ''))
    supervisor = normalize_upper(row.get('SUPERVISOR', ''))
    telefono = normalize_text(row.get('TELEFONO', ''))

    if director in PLACEHOLDERS:
        placeholder_rows.append({'row_number': index, 'column': 'DIRECTOR', 'value': row.get('DIRECTOR', '')})
    if supervisor in PLACEHOLDERS:
        placeholder_rows.append({'row_number': index, 'column': 'SUPERVISOR', 'value': row.get('SUPERVISOR', '')})

    if telefono and (not PHONE_PATTERN.match(telefono) or len(telefono) < 7 or len(telefono) > 12):
        phone_rows.append({'row_number': index, 'TELEFONO': telefono, 'length': len(telefono), 'has_digits_only': PHONE_PATTERN.match(telefono) is not None})

placeholder_summary = [
    {'column': 'DIRECTOR', 'placeholder_rows': sum(1 for item in placeholder_rows if item['column'] == 'DIRECTOR')},
    {'column': 'SUPERVISOR', 'placeholder_rows': sum(1 for item in placeholder_rows if item['column'] == 'SUPERVISOR')},
    {'column': 'TELEFONO', 'placeholder_rows': sum(1 for row in rows if normalize_upper(row.get('TELEFONO', '')) in PLACEHOLDERS)},
]

print_table('Resumen de placeholders y vacios', placeholder_summary, limit=10)
print_table('Filas con placeholders en nombre', placeholder_rows, limit=20)
print_table('Telefonos con formato sospechoso', phone_rows, limit=20)

placeholder_export = export_table_csv(placeholder_rows, 'reporte_placeholders_nombres.csv')
phone_export = export_table_csv(phone_rows, 'reporte_telefonos_sospechosos.csv')
print(f'Archivo exportado: {placeholder_export}')
print(f'Archivo exportado: {phone_export}')

### Resumen de placeholders y vacios


### Filas con placeholders en nombre


### Telefonos con formato sospechoso


column     | placeholder_rows
---------- | ----------------
DIRECTOR   | 2024            
SUPERVISOR | 535             
TELEFONO   | 946             
row_number | column     | value
---------- | ---------- | -----
2          | DIRECTOR   | --   
54         | DIRECTOR   | --   
82         | DIRECTOR   |      
83         | DIRECTOR   |      
84         | DIRECTOR   |      
85         | DIRECTOR   |      
86         | DIRECTOR   |      
92         | DIRECTOR   |      
93         | DIRECTOR   | --   
97         | DIRECTOR   |      
98         | DIRECTOR   |      
100        | DIRECTOR   |      
104        | DIRECTOR   | ---  
105        | DIRECTOR   |      
110        | DIRECTOR   |      
110        | SUPERVISOR |      
112        | DIRECTOR   |      
112        | SUPERVISOR |      
116        | DIRECTOR   | ---  
121        | DIRECTOR   |      
... 2539 filas adicionales
row_number | TELEFONO                   | length | has_digits_only
---------- | -------------------------- | ------ | -

## 2. Document Initial Data Quality Diagnostics

Aquí se resumen los problemas más visibles por variable: faltantes, formatos inválidos, dominios inconsistentes y variaciones textuales que deberán normalizarse.

In [5]:
categorical_columns = ['NIVEL', 'SECTOR', 'AREA', 'STATUS', 'MODALIDAD', 'JORNADA', 'PLAN', 'DEPARTAMENTO', 'MUNICIPIO', 'DEPARTAMENTAL']
category_summary = []

for column in categorical_columns:
    normalized_values = [normalize_upper(row.get(column, '')) for row in rows]
    counts = Counter(value for value in normalized_values if value)
    category_summary.append({
        'column': column,
        'unique_values': len(counts),
        'top_value': counts.most_common(1)[0][0] if counts else '',
        'top_count': counts.most_common(1)[0][1] if counts else 0,
    })

print_table('Resumen de categorias', category_summary, limit=20)

validation_rows = []
for column, expected_values in EXPECTED_CATEGORICAL_DOMAINS.items():
    observed = sorted({normalize_upper(row.get(column, '')) for row in rows if normalize_upper(row.get(column, ''))})
    unexpected = [value for value in observed if value not in expected_values]
    validation_rows.append({
        'column': column,
        'expected_values': ', '.join(sorted(expected_values)),
        'unexpected_values': ', '.join(unexpected),
        'unexpected_count': len(unexpected),
    })

invalid_departamentos = sorted({
    normalize_upper(row.get('DEPARTAMENTO', ''))
    for row in rows
    if normalize_upper(row.get('DEPARTAMENTO', '')) and normalize_upper(row.get('DEPARTAMENTO', '')) not in raw_department_catalog
})

invalid_departamentales = sorted({
    normalize_upper(row.get('DEPARTAMENTAL', ''))
    for row in rows
    if normalize_upper(row.get('DEPARTAMENTAL', '')) and normalize_upper(row.get('DEPARTAMENTAL', '')) not in raw_department_catalog
})

validation_rows.extend([
    {
        'column': 'DEPARTAMENTO',
        'expected_values': f'{len(raw_department_catalog)} departamentos locales',
        'unexpected_values': ', '.join(invalid_departamentos),
        'unexpected_count': len(invalid_departamentos),
    },
    {
        'column': 'DEPARTAMENTAL',
        'expected_values': f'{len(raw_department_catalog)} departamentos locales',
        'unexpected_values': ', '.join(invalid_departamentales),
        'unexpected_count': len(invalid_departamentales),
    },
])

print_table('Validacion de catalogos y dominios', validation_rows, limit=20)
validation_export = export_table_csv(validation_rows, 'reporte_validacion_catalogos.csv')
print(f'Archivo exportado: {validation_export}')
print('Nota: el workspace no contiene catalogos oficiales de MUNICIPIO; esa validacion debe completarse con la fuente externa correspondiente.')

### Resumen de categorias


### Validacion de catalogos y dominios


column        | unique_values | top_value       | top_count
------------- | ------------- | --------------- | ---------
NIVEL         | 1             | DIVERSIFICADO   | 11867    
SECTOR        | 4             | PRIVADO         | 9891     
AREA          | 3             | URBANA          | 9461     
STATUS        | 5             | ABIERTA         | 6860     
MODALIDAD     | 2             | MONOLINGUE      | 11394    
JORNADA       | 6             | DOBLE           | 3772     
PLAN          | 13            | DIARIO(REGULAR) | 7452     
DEPARTAMENTO  | 23            | CIUDAD CAPITAL  | 2161     
MUNICIPIO     | 352           | ZONA 1          | 868      
DEPARTAMENTAL | 26            | GUATEMALA NORTE | 1489     
column        | expected_values          | unexpected_values                                                                                                                                     | unexpected_count
------------- | ------------------------ | -------------------------

## 3. Build Data Quality Tables

Las tablas exportadas desde esta notebook dejan evidencia reproducible para el proceso de limpieza y para las decisiones de normalización que seguirá Roberto.

## 4. Generate Data Cleaning Plan by Variable

El plan siguiente resume qué limpiar, cómo validarlo y qué resultado esperar para cada una de las 17 variables del archivo consolidado.

In [6]:
cleaning_plan = [
    {
        'variable': 'CODIGO',
        'issues': 'Formato invalido, faltantes y repetidos.',
        'transformations': 'Conservar como texto, limpiar espacios y validar patron ##-##-####-##.',
        'validation_rules': 'Debe cumplir el patron definido y no repetirse entre filas activas.',
        'expected_outcome': 'Identificador estable y apto para cruces.'
    },
    {
        'variable': 'DISTRITO',
        'issues': 'Valores truncados o vacios.',
        'transformations': 'Normalizar texto y validar contra catalogo local de distritos.',
        'validation_rules': 'Sin espacios sobrantes ni codigos incompletos.',
        'expected_outcome': 'Distrito consistente y comparable.'
    },
    {
        'variable': 'DEPARTAMENTO',
        'issues': 'Variantes de escritura y validacion pendiente contra catalogo externo.',
        'transformations': 'Pasar a mayusculas y comparar con catalogo local de referencia.',
        'validation_rules': 'Debe pertenecer al conjunto de departamentos detectado en los archivos fuente.',
        'expected_outcome': 'Departamentos homologados.'
    },
    {
        'variable': 'MUNICIPIO',
        'issues': 'Posibles variantes ortograficas y catalogo oficial no disponible en el workspace.',
        'transformations': 'Normalizar mayusculas, espacios y tildes; validar con catalogo oficial externo.',
        'validation_rules': 'Sin categorias nuevas tras la homologacion.',
        'expected_outcome': 'Municipios listos para validacion final.'
    },
    {
        'variable': 'ESTABLECIMIENTO',
        'issues': 'Comillas, tildes, espacios multiples y variantes ortograficas.',
        'transformations': 'Limpiar caracteres invisibles y homogeneizar formato textual sin renombrar por criterio propio.',
        'validation_rules': 'Conservar el nombre institucional correcto.',
        'expected_outcome': 'Nombre del centro educativo estable.'
    },
    {
        'variable': 'DIRECCION',
        'issues': 'Abreviaturas, puntuacion irregular y vacios.',
        'transformations': 'Normalizar espacios, puntuacion y abreviaturas evidentes.',
        'validation_rules': 'No debe quedar vacia ni con placeholders.',
        'expected_outcome': 'Direccion legible y uniforme.'
    },
    {
        'variable': 'TELEFONO',
        'issues': 'Faltantes, separadores inconsistentes, letras y longitudes raras.',
        'transformations': 'Conservar como texto y limpiar separadores; aislar telefonos multiples.',
        'validation_rules': 'Solo digitos o valores explicitamente vacios.',
        'expected_outcome': 'Contacto telefonico utilizable.'
    },
    {
        'variable': 'SUPERVISOR',
        'issues': 'Acentos, mayusculas y espacios multiples.',
        'transformations': 'Normalizar texto y eliminar placeholders.',
        'validation_rules': 'Nombre de persona sin valores genericos.',
        'expected_outcome': 'Supervisor consistente.'
    },
    {
        'variable': 'DIRECTOR',
        'issues': 'Muchos placeholders y faltantes.',
        'transformations': 'Normalizar texto, detectar placeholders y dejar vacio solo cuando no exista dato real.',
        'validation_rules': 'No usar --, --- o SIN DATO como nombres validos.',
        'expected_outcome': 'Director depurado.'
    },
    {
        'variable': 'NIVEL',
        'issues': 'Dominio reducido pero debe confirmarse.',
        'transformations': 'Homologar categorias al catalogo esperado.',
        'validation_rules': 'Solo valores permitidos por la fuente.',
        'expected_outcome': 'Nivel coherente.'
    },
    {
        'variable': 'SECTOR',
        'issues': 'Categorias potencialmente diversas o variantes de escritura.',
        'transformations': 'Estandarizar nombres equivalentes.',
        'validation_rules': 'Valores dentro del dominio permitido.',
        'expected_outcome': 'Sector uniforme.'
    },
    {
        'variable': 'AREA',
        'issues': 'Debe limitarse a urbano/rural.',
        'transformations': 'Normalizar mayusculas y tildes.',
        'validation_rules': 'Solo URBANA o RURAL.',
        'expected_outcome': 'Area validada.'
    },
    {
        'variable': 'STATUS',
        'issues': 'Estados textuales con posible variacion de escritura.',
        'transformations': 'Homologar a lista controlada de estados.',
        'validation_rules': 'No introducir estados nuevos sin soporte de fuente.',
        'expected_outcome': 'Estado institucional consistente.'
    },
    {
        'variable': 'MODALIDAD',
        'issues': 'Categorias compactas pero deben revisarse.',
        'transformations': 'Normalizar texto y revisar dominios.',
        'validation_rules': 'Conjunto controlado sin variantes innecesarias.',
        'expected_outcome': 'Modalidad depurada.'
    },
    {
        'variable': 'JORNADA',
        'issues': 'Categorias de horario con posibles variantes.',
        'transformations': 'Estandarizar etiquetas y revisar valores raros.',
        'validation_rules': 'Solo etiquetas oficiales de jornada.',
        'expected_outcome': 'Jornada uniforme.'
    },
    {
        'variable': 'PLAN',
        'issues': 'Valores equivalentes escritos de forma distinta o incompleta.',
        'transformations': 'Normalizar formato textual y consolidar equivalencias.',
        'validation_rules': 'Solo planes admitidos por el dataset.',
        'expected_outcome': 'Plan academico claro.'
    },
    {
        'variable': 'DEPARTAMENTAL',
        'issues': 'Debe coincidir con el departamento de referencia o el catalogo oficial.',
        'transformations': 'Comparar contra catalogo local y validar equivalencias.',
        'validation_rules': 'Sin discrepancias con el valor departamental esperado.',
        'expected_outcome': 'Campo territorial consistente.'
    },
]

print_table('Plan de limpieza por variable', cleaning_plan, limit=17)
cleaning_export = export_table_csv(cleaning_plan, 'reporte_plan_limpieza.csv')
print(f'Archivo exportado: {cleaning_export}')

### Plan de limpieza por variable


variable        | issues                                                                            | transformations                                                                                 | validation_rules                                                               | expected_outcome                         
--------------- | --------------------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------ | -----------------------------------------
CODIGO          | Formato invalido, faltantes y repetidos.                                          | Conservar como texto, limpiar espacios y validar patron ##-##-####-##.                          | Debe cumplir el patron definido y no repetirse entre filas activas.            | Identificador estable y apto para cruces.
DISTRITO        | Valores trunc

## 5. Conclusions and Next Steps

El notebook deja listas las tablas base de diagnóstico, las exporta a `Datos/diagnosticos/` y separa claramente lo que ya puede corregirse de lo que todavía depende de catálogos externos, sobre todo en MUNICIPIO.